# Data Dictionary Generation

This notebook generates a data dictionary linking variable names to their descriptions, units, sources, and frequencies based on the paper:

**"Daily 1 km terrain resolving maps of surface fine particulate matter for the western United States 2003–2021"**
- Paper: https://www.nature.com/articles/s41597-022-01488-y
- PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC9345996/
- Data: https://springernature.figshare.com/collections/5562330

### Data Sources Summary:
- **EPA AQS**: PM2.5 measurements from monitoring stations
- **MODIS MAIAC**: Aerosol Optical Depth (AOD) satellite data
- **CFSR**: Climate Forecast System Reanalysis meteorological data
- **NED DEM**: National Elevation Dataset terrain variables
- **MODIS LST**: Land Surface Temperature climatology
- **US Census**: Population data
- **EPA SMOKE**: 114 emission surrogate datasets

In [2]:
import pandas as pd

In [3]:
# Load the data files
pm_all = pd.read_csv("../data/pm25_data_complete_2003-2021_nodups_051922.csv", low_memory=False)
pm_fixed = pd.read_csv('../data/pm25_locs_nodups_051922.csv')

print(f"pm_all: {len(pm_all):,} rows, {len(pm_all.columns)} columns")
print(f"pm_fixed: {len(pm_fixed):,} rows, {len(pm_fixed.columns)} columns")

pm_all: 2,077,111 rows, 21 columns
pm_fixed: 975 rows, 257 columns


In [4]:
# Define variable metadata from paper for pm_all (time-varying data)
pm_all_metadata = {
    'id': ('EPA Air Quality System site identifier', 'ID', 'EPA AQS', 'static', 'Links to monitoring station'),
    'date': ('Date of observation', 'YYYYMMDD', 'EPA AQS', 'daily', 'Integer format'),
    'smogP': ('Smog Potential - static layer predicting PM2.5 accumulation under stable conditions', 'probability (0-1)', 'Derived', 'static', 'GBM model; integration of pollution sources and topography'),
    'pm25': ('Fine particulate matter concentration (aerodynamic diameter <2.5 μm)', 'μg/m³', 'EPA AQS', 'daily', 'Primary outcome variable'),
    'aot': ('Aerosol Optical Depth (infilled for missing data)', 'unitless optical measure', 'MODIS MCD19A2 MAIAC', 'daily', 'Column-integrated PM extinction measure'),
    'aot_raw': ('Aerosol Optical Depth (raw without infilling)', 'unitless optical measure', 'MODIS MCD19A2 MAIAC', 'daily', 'Original satellite retrieval'),
    'cld': ('Boundary Layer Cloud Cover', 'fraction', 'CFSR', 'daily', 'Lower cloud cover indicates stable conditions; r=-0.179 with winter PM2.5'),
    'hgt': ('Standardized Geopotential Height (700-mb)', 'standardized', 'CFSR', 'daily', 'Higher pressure indicates stable conditions; r=0.211 with winter PM2.5'),
    'longwave': ('Upward Longwave Surface Radiation Flux', 'W/m²', 'CFSR', 'daily', 'Strong radiation supports inversions; r=0.182 with winter PM2.5'),
    'rh': ('2-meter Relative Humidity', 'percent', 'CFSR', 'daily', 'Lower humidity associated with PM accumulation; r=-0.185 with winter PM2.5'),
    'tmax': ('2-meter Maximum Temperature', '°C or K', 'CFSR', 'daily', 'Relatively weak predictor; r=0.080 with winter PM2.5'),
    'wind': ('10-meter Wind Speed', 'm/s', 'CFSR', 'daily', 'Lower wind correlates with pollution trapping; r=-0.217 with winter PM2.5'),
    'cadI': ('Cold Air Drainage Intensity (daily smog intensity)', 'index', 'Derived', 'daily', 'Captures temporal dynamics of stable atmospheric conditions'),
    'log_pm25': ('Natural log of PM2.5 concentration', 'log(μg/m³)', 'Derived', 'daily', 'Log-transformed for modeling'),
    'cell': ('Grid cell identifier', 'ID', 'Derived', 'static', '1-km grid cell reference'),
    'ID': ('Composite identifier (cell + date)', 'ID', 'Derived', 'daily', 'Unique observation identifier'),
    'smogI': ('Smog Intensity - daily atmospheric stability measure', 'index', 'Derived', 'daily', 'Derived from CFSR variables; same as cadI; meteorological interaction term'),
    'll_id': ('Latitude-longitude identifier', 'ID', 'Derived', 'static', 'Links pm_all to pm_fixed; format: lon_lat'),
    'cluster': ('Spatial cluster identifier', 'ID', 'Derived', 'static', 'Geographic grouping'),
    'src': ('Data source indicator', 'code', 'Various', 'static', 'Indicates data provenance'),
    'n': ('Observation count', 'count', 'Derived', 'daily', 'Number of measurements aggregated'),
}

print(f"Defined metadata for {len(pm_all_metadata)} pm_all variables")

Defined metadata for 21 pm_all variables


In [5]:
# Define metadata for pm_fixed (static/location data)
# Key variables with explicit metadata from paper
pm_fixed_metadata = {
    'Unnamed: 0': ('Row index', 'ID', 'pandas', 'static', 'Artifact from CSV export'),
    'site.id': ('Monitoring site identifier', 'ID', 'EPA AQS', 'static', 'Format: c####'),
    'll_id': ('Latitude-longitude identifier', 'ID', 'Derived', 'static', 'Links pm_fixed to pm_all'),
    'cell': ('Grid cell identifier', 'ID', 'Derived', 'static', '1-km grid cell reference'),
    'lon': ('Longitude', 'decimal degrees', 'EPA AQS', 'static', 'WGS84 coordinate system'),
    'lat': ('Latitude', 'decimal degrees', 'EPA AQS', 'static', 'WGS84 coordinate system'),
    'cluster': ('Spatial cluster identifier', 'ID', 'Derived', 'static', 'Geographic grouping'),
    'min_d': ('Minimum distance to nearest neighbor', 'km or m', 'Derived', 'static', 'Spatial isolation measure'),
    'logpd2500g': ('Log population density (2.5km Gaussian smoothed)', 'log(persons/km²)', 'US Census 2010', 'static', 'Strongest predictor of winter PM2.5; r=0.722'),
    'geomorphic_protection_index_0p2': ('Morphometric Protection Index (2km radius)', 'index', 'NED DEM', 'static', 'Measure of terrain openness'),
    'sd50k': ('Standard deviation of elevation within 50km', 'meters', 'NED DEM', 'static', 'Terrain roughness measure'),
    'cadP_p12_gbm_fit_040620': ('Cold Air Drainage Potential / Smog Potential (GBM model)', 'probability (0-1)', 'Derived', 'static', 'GBM prediction; same as smogP; gradient boosting machine prediction'),
    'epa.sum': ('Sum of EPA emission surrogates', 'composite', 'EPA SMOKE surrogates', 'static', 'Aggregated emission indicator'),
}

# Pattern-based metadata for variable groups
pattern_metadata = {
    'minf_': ('Local minima function - vertical distance to lowest elevation within radius', 'meters', 'NED DEM', 'static', 'Negative values indicate valleys trap pollution; r=-0.499 to -0.605'),
    'tpi_': ('Topographic Position Index - elevation relative to surroundings', 'meters', 'NED DEM', 'static', 'Valley/ridge position indicator'),
    'lst_day': ('MODIS Land Surface Temperature daytime monthly normal', '°C or K', 'MODIS LST 2003-2012', 'monthly climatology', 'Month indicated by suffix (01-12)'),
    'lst_night': ('MODIS Land Surface Temperature nighttime monthly normal', '°C or K', 'MODIS LST 2003-2012', 'monthly climatology', 'Month indicated by suffix (01-12)'),
}

# EPA SMOKE emission surrogate variables (from paper - 114 gridded datasets)
smoke_variables = {
    'airport_areas': 'Airport area within buffer',
    'airport_points': 'Airport point count within buffer',
    'class_1_railroad_miles': 'Class 1 railroad miles within buffer',
    'class_2_and_3_railroad_miles': 'Class 2 and 3 railroad miles within buffer',
    'commercial_land': 'Commercial land area',
    'commercial_plus_industrial': 'Commercial plus industrial land area',
    'commercial_plus_industrial_plus_institutional': 'Commercial/industrial/institutional land area',
    'commercial_plus_institutional_land': 'Commercial plus institutional land area',
    'commercial_plus_residential': 'Commercial plus residential land area',
    'commercial_timber': 'Commercial timber area',
    'county_area': 'County area',
    'crop_land': 'Agricultural crop land area',
    'drycleaners': 'Dry cleaner locations',
    'education': 'Educational facility count/area',
    'ertac_rail_yards': 'ERTAC rail yard locations',
    'extended_idle_locations': 'Extended idle truck locations',
    'exurban_housing': 'Exurban housing units',
    'food._drug._chemical_industrial_ind3': 'Food/drug/chemical industrial area',
    'forest_land': 'Forest land area',
    'gas_production_at_all_wells': 'Natural gas production at all wells',
    'gas_production_at_cbm_wells': 'Gas production at coalbed methane wells',
    'gas_production_at_gas_wells': 'Gas production at gas wells',
    'gas_stations': 'Gas station count',
    'golf_courses': 'Golf course area',
    'great_lakes_tug_zone_area': 'Great Lakes tug zone area',
    'gulf_shipping_lanes': 'Gulf shipping lane area',
    'heavy_and_high_tech_industrial_ind1.ind5': 'Heavy and high-tech industrial area',
    'heavy_industrial_ind1': 'Heavy industrial area (r=0.345 with PM2.5)',
    'high_intensity_residential': 'High intensity residential area',
    'hospital_com6': 'Hospital count/area',
    'housing_change': 'Housing unit change',
    'housing': 'Housing unit count',
    'industrial_land': 'Industrial land area',
    'industrial_plus_institutional_plus_hospitals': 'Industrial/institutional/hospital area',
    'intercity_bus_terminals': 'Intercity bus terminal count',
    'light_and_high_tech_industrial_ind2.ind5': 'Light and high-tech industrial area',
    'light_industrial_ind2': 'Light industrial area',
    'low_intensity_residential': 'Low intensity residential area',
    'marine_ports': 'Marine port area/count',
    'med_intensity_residential': 'Medium intensity residential area',
    'medical_office_clinic_com7': 'Medical office/clinic count',
    'metals_and_minerals_industrial_ind4': 'Metals and minerals industrial area',
    'military_airports': 'Military airport count/area',
    'mines': 'Mine locations',
    'navigable_waterway_activity': 'Navigable waterway activity',
    'navigable_waterway_miles': 'Navigable waterway miles',
    'ntad_amtrak_railroad_density': 'NTAD Amtrak railroad density',
    'ntad_class_1_2_3_railroad_density': 'NTAD Class 1/2/3 railroad density',
    'ntad_commuter_railroad_density': 'NTAD commuter railroad density',
    'ntad_total_railroad_density': 'NTAD total railroad density',
    'off.network_long.haul_trucks': 'Off-network long-haul truck activity (r=0.340 with PM2.5)',
    'off.network_short.haul_trucks': 'Off-network short-haul truck activity',
    'offshore_shipping_area': 'Offshore shipping area',
    'offshore_shipping_nei2011_nox': 'Offshore shipping NOx emissions (NEI 2011)',
    'oil_and_gas_wells': 'Oil and gas well count',
    'oil_production_at_all_wells': 'Oil production at all wells',
    'oil_production_at_gas_wells': 'Oil production at gas wells',
    'oil_production_at_oil_wells': 'Oil production at oil wells',
    'open_space': 'Open space area',
    'orchards_vineyards': 'Orchard and vineyard area',
    'pasture_land': 'Pasture land area',
    'personal_repair_com3': 'Personal repair services',
    'port_areas': 'Port area',
    'ports_nei2011_nox': 'Port NOx emissions (NEI 2011)',
    'professional_technical_com4_plus_general_government_gov1': 'Professional/technical/government area',
    'quarries': 'Quarry locations',
    'refineries_and_tank_farms_and_gas_stations': 'Refineries/tank farms/gas stations',
    'refineries_and_tank_farms': 'Refinery and tank farm locations',
    'residential_heating_coal': 'Residential coal heating',
    'residential_heating_distillate_oil': 'Residential distillate oil heating',
    'residential_heating_lp_gas': 'Residential LP gas heating',
    'residential_heating_natural_gas': 'Residential natural gas heating',
    'residential_heating_wood': 'Residential wood heating',
    'residential_high_density': 'High density residential area',
    'residential.commercial.industrial.institutional.government': 'Combined developed land area',
    'retail_trade_com1_plus_personal_repair_com3': 'Retail trade plus personal repair',
    'retail_trade_com1': 'Retail trade area',
    'rural_housing': 'Rural housing units',
    'rural_land_area': 'Rural land area',
    'rural_population': 'Rural population count',
    'rural_primary_road_miles': 'Rural primary road miles',
    'rural_secondary_road_miles': 'Rural secondary road miles',
    'shipping_lanes': 'Shipping lane area',
    'single_family_residential': 'Single family residential area',
    'spud_count': 'Well spud count (new wells drilled)',
    'strip_mines_quarries': 'Strip mine and quarry area',
    'suburban_housing': 'Suburban housing units',
    'total_agriculture_without_orchards_vineyards': 'Total agriculture minus orchards/vineyards',
    'total_agriculture': 'Total agricultural land area',
    'total_railroad_miles': 'Total railroad miles',
    'total_road_miles': 'Total road miles',
    'transit_bus_terminals': 'Transit bus terminal count',
    'urban_housing': 'Urban housing units',
    'urban_population': 'Urban population count',
    'urban_primary_plus_rural_primary': 'Urban plus rural primary road miles',
    'urban_primary_road_miles': 'Urban primary road miles',
    'urban_secondary_road_miles': 'Urban secondary road miles',
    'wastewater_treatment_facilities': 'Wastewater treatment facility count',
    'well_count_all_wells': 'Total well count',
    'well_count_cbm_wells': 'Coalbed methane well count',
    'well_count_gas_wells': 'Gas well count',
    'well_count_oil_wells': 'Oil well count',
}

print(f"Defined metadata for {len(pm_fixed_metadata)} explicit pm_fixed variables")
print(f"Defined {len(pattern_metadata)} pattern-based variable groups")
print(f"Defined {len(smoke_variables)} EPA SMOKE emission surrogate variables")

Defined metadata for 13 explicit pm_fixed variables
Defined 4 pattern-based variable groups
Defined 102 EPA SMOKE emission surrogate variables


In [6]:
def get_variable_metadata(var_name, file_type):
    """Look up metadata for a variable, checking explicit definitions first, then patterns."""
    
    if file_type == 'pm_all':
        if var_name in pm_all_metadata:
            desc, units, source, freq, notes = pm_all_metadata[var_name]
            return desc, units, source, freq, notes
        return ('Unknown variable', 'unknown', 'unknown', 'unknown', '')
    
    elif file_type == 'pm_fixed':
        # Check explicit metadata first
        if var_name in pm_fixed_metadata:
            desc, units, source, freq, notes = pm_fixed_metadata[var_name]
            return desc, units, source, freq, notes
        
        # Check pattern-based metadata
        for pattern, (desc, units, source, freq, notes) in pattern_metadata.items():
            if var_name.startswith(pattern):
                # Extract radius/month info from variable name
                suffix = var_name.replace(pattern, '')
                if pattern in ['minf_', 'tpi_']:
                    full_desc = f"{desc} ({suffix}m radius)"
                else:
                    full_desc = f"{desc} (month {suffix})"
                return full_desc, units, source, freq, notes
        
        # Check SMOKE variables (including _var2 suffix)
        base_name = var_name.replace('_var2', '')
        if base_name in smoke_variables:
            desc = smoke_variables[base_name]
            if var_name.endswith('_var2'):
                desc = f"{desc} (spatial variance)"
                notes = 'Variance at alternative spatial scale'
            else:
                notes = 'EPA SMOKE emission surrogate'
            return desc, 'area/count/activity units', 'EPA SMOKE surrogates', 'static', notes
        
        return ('Unknown variable', 'unknown', 'unknown', 'static', '')

# Test the function
test_vars = ['pm25', 'minf_5000', 'lst_day07', 'heavy_industrial_ind1', 'housing_var2']
print("Testing metadata lookup:")
for var in test_vars:
    file_type = 'pm_all' if var == 'pm25' else 'pm_fixed'
    meta = get_variable_metadata(var, file_type)
    print(f"  {var}: {meta[0][:50]}...")

Testing metadata lookup:
  pm25: Fine particulate matter concentration (aerodynamic...
  minf_5000: Local minima function - vertical distance to lowes...
  lst_day07: MODIS Land Surface Temperature daytime monthly nor...
  heavy_industrial_ind1: Heavy industrial area (r=0.345 with PM2.5)...
  housing_var2: Housing unit count (spatial variance)...


In [7]:
# Build the data dictionary from actual column names
rows = []

# Process pm_all columns
for col in pm_all.columns:
    desc, units, source, freq, notes = get_variable_metadata(col, 'pm_all')
    rows.append({
        'file': 'pm_all',
        'variable': col,
        'description': desc,
        'units': units,
        'source': source,
        'frequency': freq,
        'notes': notes
    })

# Process pm_fixed columns
for col in pm_fixed.columns:
    desc, units, source, freq, notes = get_variable_metadata(col, 'pm_fixed')
    rows.append({
        'file': 'pm_fixed',
        'variable': col,
        'description': desc,
        'units': units,
        'source': source,
        'frequency': freq,
        'notes': notes
    })

# Create DataFrame
data_dict = pd.DataFrame(rows)

print(f"Data dictionary created with {len(data_dict)} variables:")
print(f"  - pm_all: {len(data_dict[data_dict['file'] == 'pm_all'])} variables")
print(f"  - pm_fixed: {len(data_dict[data_dict['file'] == 'pm_fixed'])} variables")
print(f"\nUnknown variables: {len(data_dict[data_dict['description'] == 'Unknown variable'])}")
data_dict.head(10)

Data dictionary created with 278 variables:
  - pm_all: 21 variables
  - pm_fixed: 257 variables

Unknown variables: 0


,file,variable,description,units,source,frequency,notes
0,pm_all,id,EPA Air Quality System site identifier,ID,EPA AQS,static,Links to monitoring station
1,pm_all,date,Date of observation,YYYYMMDD,EPA AQS,daily,Integer format
2,pm_all,smogP,Smog Potential - static layer predicting PM2.5...,probability (0-1),Derived,static,GBM model; integration of pollution sources an...
3,pm_all,pm25,Fine particulate matter concentration (aerodyn...,μg/m³,EPA AQS,daily,Primary outcome variable
4,pm_all,aot,Aerosol Optical Depth (infilled for missing data),unitless optical measure,MODIS MCD19A2 MAIAC,daily,Column-integrated PM extinction measure
5,pm_all,aot_raw,Aerosol Optical Depth (raw without infilling),unitless optical measure,MODIS MCD19A2 MAIAC,daily,Original satellite retrieval
6,pm_all,cld,Boundary Layer Cloud Cover,fraction,CFSR,daily,Lower cloud cover indicates stable conditions;...
7,pm_all,hgt,Standardized Geopotential Height (700-mb),standardized,CFSR,daily,Higher pressure indicates stable conditions; r...
8,pm_all,longwave,Upward Longwave Surface Radiation Flux,W/m²,CFSR,daily,Strong radiation supports inversions; r=0.182 ...
9,pm_all,rh,2-meter Relative Humidity,percent,CFSR,daily,Lower humidity associated with PM accumulation...


In [8]:
# Show any unknown variables that need metadata added
unknown = data_dict[data_dict['description'] == 'Unknown variable']
if len(unknown) > 0:
    print("Variables without metadata definitions:")
    for _, row in unknown.iterrows():
        print(f"  {row['file']}: {row['variable']}")
else:
    print("All variables have metadata definitions!")

All variables have metadata definitions!


In [9]:
# Save data dictionary to CSV
data_dict.to_csv('data_dictionary.csv', index=False)
print("Data dictionary saved to: data_dictionary.csv")

# Display summary by source
print("\nVariables by data source:")
print(data_dict.groupby('source')['variable'].count().sort_values(ascending=False))

Data dictionary saved to: data_dictionary.csv

Variables by data source:
source
EPA SMOKE surrogates    205
MODIS LST 2003-2012      24
NED DEM                  18
Derived                  14
CFSR                      6
EPA AQS                   6
MODIS MCD19A2 MAIAC       2
US Census 2010            1
Various                   1
pandas                    1
Name: variable, dtype: int64


In [10]:
# Display the full data dictionary (first 30 rows)
print("Data Dictionary Preview (first 30 rows):")
print("=" * 100)
data_dict.head(30)

Data Dictionary Preview (first 30 rows):


,file,variable,description,units,source,frequency,notes
0,pm_all,id,EPA Air Quality System site identifier,ID,EPA AQS,static,Links to monitoring station
1,pm_all,date,Date of observation,YYYYMMDD,EPA AQS,daily,Integer format
2,pm_all,smogP,Smog Potential - static layer predicting PM2.5...,probability (0-1),Derived,static,GBM model; integration of pollution sources an...
3,pm_all,pm25,Fine particulate matter concentration (aerodyn...,μg/m³,EPA AQS,daily,Primary outcome variable
4,pm_all,aot,Aerosol Optical Depth (infilled for missing data),unitless optical measure,MODIS MCD19A2 MAIAC,daily,Column-integrated PM extinction measure
5,pm_all,aot_raw,Aerosol Optical Depth (raw without infilling),unitless optical measure,MODIS MCD19A2 MAIAC,daily,Original satellite retrieval
6,pm_all,cld,Boundary Layer Cloud Cover,fraction,CFSR,daily,Lower cloud cover indicates stable conditions;...
7,pm_all,hgt,Standardized Geopotential Height (700-mb),standardized,CFSR,daily,Higher pressure indicates stable conditions; r...
8,pm_all,longwave,Upward Longwave Surface Radiation Flux,W/m²,CFSR,daily,Strong radiation supports inversions; r=0.182 ...
9,pm_all,rh,2-meter Relative Humidity,percent,CFSR,daily,Lower humidity associated with PM accumulation...


In [13]:
# Highlight all derived features
derived = data_dict[data_dict['source'] == 'Derived'].copy()
print(f"Total derived features: {len(derived)}\n")
derived

Total derived features: 14



,file,variable,description,units,source,frequency,notes
2,pm_all,smogP,Smog Potential - static layer predicting PM2.5...,probability (0-1),Derived,static,GBM model; integration of pollution sources an...
12,pm_all,cadI,Cold Air Drainage Intensity (daily smog intens...,index,Derived,daily,Captures temporal dynamics of stable atmospher...
13,pm_all,log_pm25,Natural log of PM2.5 concentration,log(μg/m³),Derived,daily,Log-transformed for modeling
14,pm_all,cell,Grid cell identifier,ID,Derived,static,1-km grid cell reference
15,pm_all,ID,Composite identifier (cell + date),ID,Derived,daily,Unique observation identifier
16,pm_all,smogI,Smog Intensity - daily atmospheric stability m...,index,Derived,daily,Derived from CFSR variables; same as cadI; met...
17,pm_all,ll_id,Latitude-longitude identifier,ID,Derived,static,Links pm_all to pm_fixed; format: lon_lat
18,pm_all,cluster,Spatial cluster identifier,ID,Derived,static,Geographic grouping
20,pm_all,n,Observation count,count,Derived,daily,Number of measurements aggregated
23,pm_fixed,ll_id,Latitude-longitude identifier,ID,Derived,static,Links pm_fixed to pm_all
